<a href="https://colab.research.google.com/github/CJadlaon/Kimi_Agent/blob/dev/_logstripperAWS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import re

def sanitize_log_for_llm(raw_log: str) -> str:
    if not raw_log or not isinstance(raw_log, str):
        return ""
    clean_log = re.sub(r'\x1b\[[0-9;]*m', '', raw_log)
    clean_log = re.sub(r'\{\s*"timestamp":\s*"\d{4}-\d{2}-\d{2}T[\d:.Z]+"\s*\}', '', clean_log)
    clean_log = re.sub(r'\[?\d{4}-\d{2}-\d{2}[T\s]\d{2}:\d{2}:\d{2}(?:\.\d{3})?(?:Z|\s\+\d{4})?\]?', '', clean_log)
    clean_log = re.sub(r'^.*HIPAA_AUDIT.*$', '', clean_log, flags=re.MULTILINE)
    clean_log = re.sub(r'^.*\[sessionPool\].*$', '', clean_log, flags=re.MULTILINE)

    def format_level(match):
        level = match.group(1)
        return '' if level == 'info' else f"{level.upper()}: "

    clean_log = re.sub(r'^(info|warn|error):\s+', format_level, clean_log, flags=re.MULTILINE)
    clean_log = re.sub(r'\[server\]\s*', '', clean_log)
    clean_log = re.sub(r'ip=::ffff:\d{1,3}(?:\.\d{1,3}){3}\s*', '', clean_log)
    clean_log = re.sub(r'org(?:Id)?=[a-z0-9]+\s*', '', clean_log)
    clean_log = re.sub(r'\{\s*\}', '', clean_log)
    clean_log = re.sub(r'\n\s*\n', '\n', clean_log)
    return clean_log.strip()

# --- Paste your raw logs here ---
raw_aws_log_text = """
info: [server] POST /run
{
    "patientId": "P-1782913193302-2",
    "systems": [
        "careficient"
    ],
    "timestamp": "2026-07-09T16:41:03.090Z"
}
info: HIPAA_AUDIT action=AGENT_RUN patientId=P-1782913193302-2 orgId=cmqk0f98y0001thyexv93hbje ip=::ffff:169.254.172.3
{
    "timestamp": "2026-07-09T16:41:03.090Z"
}
info: [server] Starting run for patient=P-1782913193302-2 org=cmqk0f98y0001thyexv93hbje (1/4 global, 1/2 org)
{
    "timestamp": "2026-07-09T16:41:03.091Z"
}
info: [run] Starting Stagehand extraction for patient=P-1782913193302-2 (pt:a48892d3) systems=[careficient]
{
    "timestamp": "2026-07-09T16:41:03.091Z"
}
info: [audit] run.start  ok
{
    "action": "run.start",
    "actor": "agent",
    "audit": "v1",
    "durationMs": null,
    "meta": {
        "patientNamePresent": true,
        "systems": [
            "careficient"
        ]
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "ok",
    "target": null,
    "timestamp": "2026-07-09T16:41:03.091Z",
    "ts": "2026-07-09T16:41:03.091Z",
    "vendor": null
}
info: [Stagehand] Initialized (LOCAL, model=bedrock/us.anthropic.claude-sonnet-4-6, act/extract retry-wrapped)
{
    "timestamp": "2026-07-09T16:41:03.886Z"
}
info: [sessionPool] Created session for cmqk0f98y0001thyexv93hbje:careficient (size=1/10)
{
    "timestamp": "2026-07-09T16:41:03.886Z"
}
info: [CF] In-memory extraction for pt:a48892d3
{
    "timestamp": "2026-07-09T16:41:03.889Z"
}
info: [CF] Login attempt — username=present(17ch), password=present(14ch)
{
    "timestamp": "2026-07-09T16:41:05.065Z"
}
info: [CF] Logged in (attempt 1/3)
{
    "timestamp": "2026-07-09T16:41:10.203Z"
}
info: [CF] openPatient: deterministic search for pt:a48892d3
{
    "timestamp": "2026-07-09T16:41:10.207Z"
}
info: [CF] openPatient: last=pt:fb102c43 first=pt:94aec9fb
{
    "timestamp": "2026-07-09T16:41:10.207Z"
}
info: [CF] openPatient: 1 candidate row(s) via encounter-detail
{
    "timestamp": "2026-07-09T16:41:14.151Z"
}
info: [CF] openPatient: matched pt:a5008abe (100% exact_canonical)
{
    "timestamp": "2026-07-09T16:41:14.152Z"
}
info: [CF] Patient record opened: pt:ac0c45da
{
    "timestamp": "2026-07-09T16:41:26.493Z"
}
info: [CF] Patient page verified open (fact-report-visible)
{
    "timestamp": "2026-07-09T16:41:26.499Z"
}
info: [CF] Downloading Patient Fact Report...
{
    "timestamp": "2026-07-09T16:41:26.499Z"
}
warn: [CF] Fact Report fetch failed: no-eventtarget buttonHTML="<button type="button" class="rwOkBtn" onclick="$find('{0}').close(true); return false;">##LOC[OK]##</button>"
{
    "timestamp": "2026-07-09T16:41:36.796Z"
}
warn: [CF] Patient Fact Report — retry 1/1 after 2000ms (Patient Fact Report: returned no file)
{
    "timestamp": "2026-07-09T16:41:36.796Z"
}
warn: [CF] Fact Report fetch failed: no-eventtarget buttonHTML="<button type="button" class="rwOkBtn" onclick="$find('{0}').close(true); return false;">##LOC[OK]##</button>"
{
    "timestamp": "2026-07-09T16:41:38.801Z"
}
warn: [CF] Patient Fact Report extraction failed: Patient Fact Report: returned no file
{
    "timestamp": "2026-07-09T16:41:38.801Z"
}
info: [CF] Dismissed 1 modal(s) (likely session-timeout / "Are you still there?")
{
    "timestamp": "2026-07-09T16:41:38.806Z"
}
info: [CF] Downloading Orders PDF...
{
    "timestamp": "2026-07-09T16:41:40.307Z"
}
warn: [CF] Orders PDF export failed: html-response status=200 ct=text/html; charset=utf-8 len=311610 snippet=" <!DOCTYPE html> <html lang="en" > <head id="ctl00_ctl00_MasterHeader"><meta http-equiv="Content-Type" content="text/html; charset=utf-8" /><meta name"
{
    "timestamp": "2026-07-09T16:41:45.981Z"
}
warn: [CF] Orders PDF export — retry 1/1 after 2000ms (Orders PDF export: returned no file)
{
    "timestamp": "2026-07-09T16:41:45.981Z"
}
warn: [CF] Orders PDF export failed: html-response status=200 ct=text/html; charset=utf-8 len=311610 snippet=" <!DOCTYPE html> <html lang="en" > <head id="ctl00_ctl00_MasterHeader"><meta http-equiv="Content-Type" content="text/html; charset=utf-8" /><meta name"
{
    "timestamp": "2026-07-09T16:41:48.541Z"
}
warn: [CF] Orders PDF export failed (Orders PDF export: returned no file) — falling back to DOM scrape
{
    "timestamp": "2026-07-09T16:41:48.541Z"
}
info: [CF] Orders DOM fallback: 485=true, F2F=false, orders=22
{
    "timestamp": "2026-07-09T16:41:48.553Z"
}
info: [CF] Downloading Service Notes PDF...
{
    "timestamp": "2026-07-09T16:41:48.553Z"
}
warn: [CF] Service Notes PDF export failed: html-response status=200 ct=text/html; charset=utf-8 len=676831 snippet=" <!DOCTYPE html> <html lang="en" > <head id="ctl00_ctl00_MasterHeader"><meta http-equiv="Content-Type" content="text/html; charset=utf-8" /><meta name"
{
    "timestamp": "2026-07-09T16:41:53.635Z"
}
warn: [CF] Service Notes PDF export — retry 1/1 after 2000ms (Service Notes PDF export: returned no file)
{
    "timestamp": "2026-07-09T16:41:53.635Z"
}
warn: [CF] Service Notes PDF export failed: html-response status=200 ct=text/html; charset=utf-8 len=676831 snippet=" <!DOCTYPE html> <html lang="en" > <head id="ctl00_ctl00_MasterHeader"><meta http-equiv="Content-Type" content="text/html; charset=utf-8" /><meta name"
{
    "timestamp": "2026-07-09T16:41:55.975Z"
}
warn: [CF] Service Notes PDF export failed (Service Notes PDF export: returned no file) — falling back to calendar DOM scrape
{
    "timestamp": "2026-07-09T16:41:55.975Z"
}
warn: [CF] Tab "Scheduling" did not activate after click — extraction may read wrong tab
{
    "timestamp": "2026-07-09T16:42:09.629Z"
}
info: [CF] Dismissed 1 modal(s) (likely session-timeout / "Are you still there?")
{
    "timestamp": "2026-07-09T16:42:33.896Z"
}
info: [CF] Service Notes DOM fallback: 0 visits (none)
{
    "timestamp": "2026-07-09T16:42:41.663Z"
}
info: [CF] Scanning Docs tab...
{
    "timestamp": "2026-07-09T16:42:41.663Z"
}
info: [CF] Docs: 6 documents
{
    "timestamp": "2026-07-09T16:42:56.186Z"
}
info: [CF] Scanning Scheduling tab...
{
    "timestamp": "2026-07-09T16:42:56.186Z"
}
warn: [CF] Tab "Scheduling" did not activate after click — extraction may read wrong tab
{
    "timestamp": "2026-07-09T16:43:08.740Z"
}
info: [server] POST /run
{
    "patientId": "P-1782913193302-2",
    "systems": [
        "inovalon",
        "vonage",
        "tigerconnect"
    ],
    "timestamp": "2026-07-09T16:43:18.145Z"
}
info: HIPAA_AUDIT action=AGENT_RUN patientId=P-1782913193302-2 orgId=cmqk0f98y0001thyexv93hbje ip=::ffff:169.254.172.3
{
    "timestamp": "2026-07-09T16:43:18.148Z"
}
info: [server] Starting run for patient=P-1782913193302-2 org=cmqk0f98y0001thyexv93hbje (2/4 global, 2/2 org)
{
    "timestamp": "2026-07-09T16:43:18.149Z"
}
info: [run] Starting Stagehand extraction for patient=P-1782913193302-2 (pt:a48892d3) systems=[inovalon, vonage, tigerconnect]
{
    "timestamp": "2026-07-09T16:43:18.150Z"
}
info: [audit] run.start  ok
{
    "action": "run.start",
    "actor": "agent",
    "audit": "v1",
    "durationMs": null,
    "meta": {
        "patientNamePresent": true,
        "systems": [
            "inovalon",
            "vonage",
            "tigerconnect"
        ]
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "ok",
    "target": null,
    "timestamp": "2026-07-09T16:43:18.150Z",
    "ts": "2026-07-09T16:43:18.150Z",
    "vendor": null
}
info: [Stagehand] Initialized (LOCAL, model=bedrock/us.anthropic.claude-sonnet-4-6, act/extract retry-wrapped)
{
    "timestamp": "2026-07-09T16:43:19.078Z"
}
info: [sessionPool] Created session for cmqk0f98y0001thyexv93hbje:inovalon (size=2/10)
{
    "timestamp": "2026-07-09T16:43:19.078Z"
}
info: [IN] Starting extraction for pt:a48892d3 (<no-mbi>) at https://apps.inovalon.com/ — username=present, password=present
{
    "timestamp": "2026-07-09T16:43:19.079Z"
}
info: [IN] Restored 6 cookie(s) — attempting login skip
{
    "timestamp": "2026-07-09T16:43:19.105Z"
}
info: [IN] Restored cookies stale — invalidating, falling through to credentials
{
    "timestamp": "2026-07-09T16:43:24.479Z"
}
info: [IN] Two-step login detected, clicking Continue
{
    "timestamp": "2026-07-09T16:43:26.963Z"
}
warn: [CF] Tab "Schedule" did not activate after click — extraction may read wrong tab
{
    "timestamp": "2026-07-09T16:43:27.385Z"
}
info: [IN] Login successful — patient search page loaded
{
    "timestamp": "2026-07-09T16:43:43.112Z"
}
info: [vendor-session] saved 6 cookie(s) for inovalon
{
    "timestamp": "2026-07-09T16:43:43.149Z"
}
info: [IN] Checking eligibility...
{
    "timestamp": "2026-07-09T16:43:55.669Z"
}
info: [IN] Extracting claims...
{
    "timestamp": "2026-07-09T16:43:59.504Z"
}
info: [CF] Dismissed 1 modal(s) (likely session-timeout / "Are you still there?")
{
    "timestamp": "2026-07-09T16:44:03.896Z"
}
warn: [retryStagehand] extract attempt 1/3 failed (transient=false): No object generated: response did not match schema.
{
    "timestamp": "2026-07-09T16:44:05.961Z"
}
warn: [CF] Scheduling extract failed: No object generated: response did not match schema.
{
    "timestamp": "2026-07-09T16:44:05.961Z"
}
info: [CF] Scanning Clinical tab...
{
    "timestamp": "2026-07-09T16:44:05.961Z"
}
[2026-07-09 16:44:05.960 +0000] [31mERROR[39m: [36mNo object generated: response did not match schema.[39m
    [35mcategory[39m: "AISDK error"
    [35mcause[39m: {
      "name": "AI_TypeValidationError",
      "cause": {
        "issues": [
          {
            "code": "invalid_type",
            "expected": "boolean",
            "received": "string",
            "path": [
              "futurePlotsDeleted"
            ],
            "message": "Expected boolean, received string"
          },
          {
            "code": "invalid_type",
            "expected": "number",
            "received": "string",
            "path": [
              "totalVisits"
            ],
            "message": "Expected number, received string"
          },
          {
            "code": "invalid_type",
            "expected": "number",
            "received": "string",
            "path": [
              "rnVisits"
            ],
            "message": "Expected number, received string"
          },
          {
            "code": "invalid_type",
            "expected": "number",
            "received": "string",
            "path": [
              "lpnVisits"
            ],
            "message": "Expected number, received string"
          },
          {
            "code": "invalid_type",
            "expected": "number",
            "received": "string",
            "path": [
              "hhaVisits"
            ],
            "message": "Expected number, received string"
          },
          {
            "code": "invalid_type",
            "expected": "number",
            "received": "string",
            "path": [
              "ptVisits"
            ],
            "message": "Expected number, received string"
          },
          {
            "code": "invalid_type",
            "expected": "number",
            "received": "string",
            "path": [
              "otVisits"
            ],
            "message": "Expected number, received string"
          },
          {
            "code": "invalid_type",
            "expected": "number",
            "received": "string",
            "path": [
              "mswVisits"
            ],
            "message": "Expected number, received string"
          },
          {
            "code": "invalid_type",
            "expected": "number",
            "received": "string",
            "path": [
              "oasisVisits"
            ],
            "message": "Expected number, received string"
          },
          {
            "code": "invalid_type",
            "expected": "number",
            "received": "string",
            "path": [
              "rocVisits"
            ],
            "message": "Expected number, received string"
          }
        ],
        "name": "ZodError"
      },
      "value": {
        "certEnd": "08/01/2026",
        "certStart": "06/03/2026",
        "futurePlotsDeleted": "null",
        "hhaVisits": "null",
        "hospitalAdmitDate": "null",
        "hospitalDischargeDate": "null",
        "hospitalStatus": "Active",
        "lpnVisits": "null",
        "mswVisits": "null",
        "nextScheduledVisit": "null",
        "oasisVisits": "null",
        "otVisits": "null",
        "ptVisits": "null",
        "rnVisits": "null",
        "rocVisits": "null",
        "totalVisits": "null"
      }
    }
    [35mtext[39m: "{\"certEnd\":\"08/01/2026\",\"certStart\":\"06/03/2026\",\"futurePlotsDeleted\":\"null\",\"hhaVisits\":\"null\",\"hospitalAdmitDate\":\"null\",\"hospitalDischargeDate\":\"null\",\"hospitalStatus\":\"Active\",\"lpnVisits\":\"null\",\"mswVisits\":\"null\",\"nextScheduledVisit\":\"null\",\"oasisVisits\":\"null\",\"otVisits\":\"null\",\"ptVisits\":\"null\",\"rnVisits\":\"null\",\"rocVisits\":\"null\",\"totalVisits\":\"null\"}"
    [35mresponse[39m: {
      "id": "aiobj-QRu0S0WmCaK5LMjXuWHYdghB",
      "timestamp": "2026-07-09T16:44:05.958Z",
      "modelId": "us.anthropic.claude-sonnet-4-6",
      "headers": {
        "connection": "keep-alive",
        "content-length": "767",
        "content-type": "application/json",
        "date": "Thu, 09 Jul 2026 16:44:05 GMT",
        "x-amzn-requestid": "0b7485a7-940a-4759-8f62-82dd47eb8bc6"
      }
    }
    [35musage[39m: {
      "inputTokens": 8303,
      "outputTokens": 345,
      "totalTokens": 8648,
      "cachedInputTokens": 0
    }
    [35mfinishReason[39m: "stop"
info: [IN] Extracting payment data...
{
    "timestamp": "2026-07-09T16:44:09.752Z"
}
info: [CF] Clinical: DME items=0, secondaryDx=0
{
    "timestamp": "2026-07-09T16:44:15.800Z"
}
info: [CF] Scanning Meds tab...
{
    "timestamp": "2026-07-09T16:44:15.800Z"
}
info: [IN] Checking for denied claims...
{
    "timestamp": "2026-07-09T16:44:19.615Z"
}
info: [CF] Meds: 5 entries
{
    "timestamp": "2026-07-09T16:44:27.110Z"
}
info: [CF] Scanning Payers tab for MBI...
{
    "timestamp": "2026-07-09T16:44:27.111Z"
}
info: [CF] Dismissed 1 modal(s) (likely session-timeout / "Are you still there?")
{
    "timestamp": "2026-07-09T16:44:27.118Z"
}
info: [IN] Extracted 0 claims, 0 denied, duplicateHH=unknown
{
    "timestamp": "2026-07-09T16:44:30.251Z"
}
info: [validator] ═══ INOVALON COVERAGE — 4/7 fields present (single run) ═══
{
    "timestamp": "2026-07-09T16:44:30.264Z"
}
warn: [validator] ❌ INOVALON fields absent: duplicateHH, otherActiveHH, claims.count
{
    "timestamp": "2026-07-09T16:44:30.264Z"
}
info: [validator] inovalon.eligibilityStatus: 1
{
    "timestamp": "2026-07-09T16:44:30.264Z"
}
info: [validator] inovalon.mbiVerified: 1
{
    "timestamp": "2026-07-09T16:44:30.265Z"
}
info: [validator] inovalon.duplicateHH: 0
{
    "timestamp": "2026-07-09T16:44:30.265Z"
}
info: [validator] inovalon.otherActiveHH: 0
{
    "timestamp": "2026-07-09T16:44:30.265Z"
}
info: [validator] inovalon.claims.count: 0
{
    "timestamp": "2026-07-09T16:44:30.265Z"
}
info: [validator] inovalon.deniedClaims: 1
{
    "timestamp": "2026-07-09T16:44:30.265Z"
}
info: [validator] inovalon.pendingClaims: 1
{
    "timestamp": "2026-07-09T16:44:30.265Z"
}
info: [validator] ═══ END INOVALON COVERAGE ═══
{
    "timestamp": "2026-07-09T16:44:30.265Z"
}
info: [inovalon] done (Stagehand) — 2 findings, session fresh
{
    "timestamp": "2026-07-09T16:44:30.265Z"
}
info: [audit] extract inovalon ok
{
    "action": "extract",
    "actor": "agent",
    "audit": "v1",
    "durationMs": 72109,
    "meta": {
        "failCount": 2,
        "sessionFresh": true,
        "transport": "stagehand"
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "ok",
    "target": "stagehand",
    "timestamp": "2026-07-09T16:44:30.266Z",
    "ts": "2026-07-09T16:44:30.266Z",
    "vendor": "inovalon"
}
info: [Stagehand] Initialized (LOCAL, model=bedrock/us.anthropic.claude-sonnet-4-6, act/extract retry-wrapped)
{
    "timestamp": "2026-07-09T16:44:31.171Z"
}
info: [sessionPool] Created session for cmqk0f98y0001thyexv93hbje:vonage (size=3/10)
{
    "timestamp": "2026-07-09T16:44:31.171Z"
}
info: [VG] Starting extraction for pt:a48892d3 at https://app.vonage.com/onage.com/ — username=present, password=present
{
    "timestamp": "2026-07-09T16:44:31.172Z"
}
info: [VG] Two-step login detected, clicking Continue/Next
{
    "timestamp": "2026-07-09T16:44:42.674Z"
}
warn: [CF] Payers: rejecting suspect MBI mbi:74234e98 (likely hallucinated)
{
    "timestamp": "2026-07-09T16:44:52.338Z"
}
info: [CF] Payers: MBI=not found, Part A=Active, Part B=null
{
    "timestamp": "2026-07-09T16:44:52.338Z"
}
info: [CF] MBI source: none
{
    "timestamp": "2026-07-09T16:44:52.338Z"
}
info: [recovery] Screenshot: /tmp/carepro-careficient-payers-no-mbi-MARIA_VIOLETA_APAT-1783615492341.png
{
    "timestamp": "2026-07-09T16:44:52.466Z"
}
info: [CF] Complete for pt:a48892d3 — 12 findings
{
    "timestamp": "2026-07-09T16:44:52.466Z"
}
warn: [adapter:careficient] content quality: 1 reject(s) [FV-IDENTITY-PRESENT:demographics], 1 warning(s) [CF-ORDERS-IMPLY-EPISODE:scheduling]
{
    "timestamp": "2026-07-09T16:44:52.468Z"
}
info: [careficient] done (Stagehand) — 12 findings, session fresh
{
    "timestamp": "2026-07-09T16:44:52.468Z"
}
info: [audit] extract careficient ok
{
    "action": "extract",
    "actor": "agent",
    "audit": "v1",
    "durationMs": 229373,
    "meta": {
        "failCount": 12,
        "sessionFresh": true,
        "transport": "stagehand"
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "ok",
    "target": "stagehand",
    "timestamp": "2026-07-09T16:44:52.468Z",
    "ts": "2026-07-09T16:44:52.468Z",
    "vendor": "careficient"
}
warn: [VG] Still on login page after 5s — waiting another 5s for slow redirect
{
    "timestamp": "2026-07-09T16:44:58.607Z"
}
error: [VG] Login failed — page still on login after submit. Diagnostics:
{
    "url": "https://login.auth.vonage.com/authenticationendpoint/login-init.do?client_id=d119f6e8ca24442ca0f2a17fc46dcd0c&code_challenge=W4wn3pd0Kh0wO-R6hPn-clQ7f5z8cHMLbnvA4HEd5mc&code_challenge_method=S256&redirect_uri=https%3A%2F%2Fapp.vonage.com%2Flogin%3Fsso%3Dtrue&response_type=code&scope=openid&sessionDataKey=8MgX9CwCInpMeDRK7NkVRgDNxgpqAO_Puv-BEO_jEn5&sp=web",
    "title": "Login",
    "messages": []
}

{
    "timestamp": "2026-07-09T16:45:03.614Z"
}
info: [recovery] Screenshot: /tmp/carepro-vg-login-failed-1783615503616.png
{
    "timestamp": "2026-07-09T16:45:05.750Z"
}
error: [vonage] Stagehand agent failed: Vonage login failed — still on login page after 10s. Check credentials, MFA, or session lockout.
Error: Vonage login failed — still on login page after 10s. Check credentials, MFA, or session lockout.
    at Object.run (/app/src/stagehand/vonage.js:172:11)
    at async withDetachedFrameRetry (/app/src/utils/frameRecovery.js:66:14)
    at async run (/app/src/index.js:235:29)
    at async /app/src/server.js:339:21
{
    "timestamp": "2026-07-09T16:45:05.751Z"
}
info: [audit] extract vonage error
{
    "action": "extract",
    "actor": "agent",
    "audit": "v1",
    "durationMs": 35482,
    "meta": {
        "errorMessage": "Vonage login failed — still on login page after 10s. Check credentials, MFA, or session lockout.",
        "errorStack": "Error: Vonage login failed — still on login page after 10s. Check credentials, MFA, or session lockout.\n    at Object.run (/app/src/stagehand/vonage.js:172:11)\n    at async withDetachedFrameRetry (/app/src/utils/frameRecovery.js:66:14)\n    at async run (/app/src/index.js:235:29)\n    at async /app/src/server.js:339:21",
        "transport": "stagehand"
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "error",
    "target": "stagehand",
    "timestamp": "2026-07-09T16:45:05.751Z",
    "ts": "2026-07-09T16:45:05.751Z",
    "vendor": "vonage"
}
info: [sessionPool] Closed cmqk0f98y0001thyexv93hbje:vonage (invalidate)
{
    "timestamp": "2026-07-09T16:45:05.773Z"
}
info: [launcher] Browser launched
{
    "timestamp": "2026-07-09T16:45:06.475Z"
}
info: [tigerconnect] Navigating to login page
{
    "timestamp": "2026-07-09T16:45:06.765Z"
}
info: [tigerconnect] Step 1: entering username
{
    "timestamp": "2026-07-09T16:45:14.194Z"
}
info: [type] Email/Username
{
    "timestamp": "2026-07-09T16:45:14.989Z"
}
info: [click] Submit Username
{
    "timestamp": "2026-07-09T16:45:15.334Z"
}
info: [tigerconnect] Step 2: entering password
{
    "timestamp": "2026-07-09T16:45:17.335Z"
}
info: [type] Password
{
    "timestamp": "2026-07-09T16:45:17.937Z"
}
info: [click] Submit Password
{
    "timestamp": "2026-07-09T16:45:18.264Z"
}
info: [tigerconnect] Login successful — messenger loaded
{
    "timestamp": "2026-07-09T16:45:29.842Z"
}
info: [tigerconnect] Searching for patient thread: lastName=pt:fb102c43 firstName=pt:94aec9fb
{
    "timestamp": "2026-07-09T16:45:29.843Z"
}
error: [Analyzer] Analysis failed: Expected ',' or ']' after array element in JSON at position 7631
{
    "timestamp": "2026-07-09T16:45:31.010Z"
}
info: [audit] llm.analyzer  error
{
    "action": "llm.analyzer",
    "actor": "agent",
    "audit": "v1",
    "durationMs": 38568,
    "meta": {
        "errorMessage": "Expected ',' or ']' after array element in JSON at position 7631"
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "error",
    "target": null,
    "timestamp": "2026-07-09T16:45:31.039Z",
    "ts": "2026-07-09T16:45:31.039Z",
    "vendor": null
}
info: [run] Results posted for P-1782913193302-2
{
    "timestamp": "2026-07-09T16:45:32.541Z"
}
info: [run] Extraction complete for P-1782913193302-2
{
    "timestamp": "2026-07-09T16:45:32.541Z"
}
info: [audit] run.end  ok
{
    "action": "run.end",
    "actor": "agent",
    "audit": "v1",
    "durationMs": null,
    "meta": {
        "hasAnalysis": false,
        "systemSummary": {
            "careficient": "ok"
        }
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "ok",
    "target": null,
    "timestamp": "2026-07-09T16:45:32.541Z",
    "ts": "2026-07-09T16:45:32.541Z",
    "vendor": null
}
info: [server] Run complete for patient=P-1782913193302-2 in 269.6s
{
    "timestamp": "2026-07-09T16:45:32.740Z"
}
info: [click] Search Icon
{
    "timestamp": "2026-07-09T16:45:44.244Z"
}
warn: [retry] Thread search attempt 1/3 failed: Thread search: attempt 1 timed out after 30000ms
{
    "timestamp": "2026-07-09T16:46:20.640Z"
}
warn: [tigerconnect] Typed "APAT" — search did not settle after 15499ms (state=populating)
{
    "timestamp": "2026-07-09T16:46:23.839Z"
}
warn: [tigerconnect] Typed "APAT" — search did not settle after 15399ms (state=populating)
{
    "timestamp": "2026-07-09T16:46:50.839Z"
}
info: [tigerconnect] No thread found by last name, trying LASTNAME, FIRSTNAME
{
    "timestamp": "2026-07-09T16:46:52.240Z"
}
warn: [retry] Thread search attempt 1/3 failed: Thread search: attempt 1 timed out after 30000ms
{
    "timestamp": "2026-07-09T16:47:22.241Z"
}
info: [tigerconnect] Search input not present — re-opening search panel
{
    "timestamp": "2026-07-09T16:47:29.241Z"
}
warn: [tigerconnect] Typed "APAT, MARIA" — search did not settle after 18198ms (state=no-sidebar)
{
    "timestamp": "2026-07-09T16:47:33.439Z"
}
warn: [retry] Thread search attempt 2/3 failed: Thread search: attempt 2 timed out after 30000ms
{
    "timestamp": "2026-07-09T16:47:54.440Z"
}
info: [click] Search Icon
{
    "timestamp": "2026-07-09T16:47:56.243Z"
}
warn: [retry] Open search attempt 1/3 failed: Open search: attempt 1 timed out after 30000ms
{
    "timestamp": "2026-07-09T16:47:59.241Z"
}
info: [click] Search Icon
{
    "timestamp": "2026-07-09T16:48:09.739Z"
}
warn: [tigerconnect] Search panel did not open, trying evaluate click
{
    "timestamp": "2026-07-09T16:48:15.640Z"
}
warn: [retry] Open search attempt 2/3 failed: Search panel still not visible after evaluate click
{
    "timestamp": "2026-07-09T16:48:24.940Z"
}
warn: [retry] Thread search attempt 3/3 failed: Thread search: attempt 3 timed out after 30000ms
{
    "timestamp": "2026-07-09T16:48:28.440Z"
}
warn: [tigerconnect] search term "pt:7d66799e" failed to run: Thread search: attempt 3 timed out after 30000ms — trying next strategy
{
    "timestamp": "2026-07-09T16:48:28.441Z"
}
info: [tigerconnect] No thread found, trying first name
{
    "timestamp": "2026-07-09T16:48:28.441Z"
}
info: [tigerconnect] Search input not present — re-opening search panel
{
    "timestamp": "2026-07-09T16:48:28.541Z"
}
warn: [tigerconnect] Typed "APAT, MARIA" — search did not settle after 15800ms (state=no-sidebar)
{
    "timestamp": "2026-07-09T16:48:30.740Z"
}
info: [click] Search Icon
{
    "timestamp": "2026-07-09T16:48:32.740Z"
}
warn: [tigerconnect] Search panel did not open, trying evaluate click
{
    "timestamp": "2026-07-09T16:48:35.139Z"
}
info: [click] Search Icon
{
    "timestamp": "2026-07-09T16:48:36.540Z"
}
warn: [retry] Open search attempt 1/3 failed: Search panel still not visible after evaluate click
{
    "timestamp": "2026-07-09T16:48:41.941Z"
}
warn: [tigerconnect] Search panel did not open, trying evaluate click
{
    "timestamp": "2026-07-09T16:48:41.941Z"
}
warn: [retry] Open search attempt 3/3 failed: Search panel still not visible after evaluate click
{
    "timestamp": "2026-07-09T16:48:46.140Z"
}
info: [click] Search Icon
{
    "timestamp": "2026-07-09T16:48:51.847Z"
}
warn: [retry] Thread search attempt 1/3 failed: Thread search: attempt 1 timed out after 30000ms
{
    "timestamp": "2026-07-09T16:48:58.442Z"
}
info: [tigerconnect] Typed "MARIA" — results ready (has-results, 3100ms)
{
    "timestamp": "2026-07-09T16:49:04.440Z"
}
info: [tigerconnect] Typed "MARIA" — results ready (has-results, 700ms)
{
    "timestamp": "2026-07-09T16:49:11.940Z"
}
info: [tigerconnect] Clicking group thread at (250, 486): HH - APAT, MARIA VIOLETA - SIR ALEX
{
    "timestamp": "2026-07-09T16:49:13.441Z"
}
info: [tigerconnect] Found thread (attempt 3 - first name): pt:cffbe708
{
    "timestamp": "2026-07-09T16:49:26.940Z"
}
info: [sessionPool] Closed cmqk0f98y0001thyexv93hbje:inovalon (idle)
{
    "timestamp": "2026-07-09T16:49:36.248Z"
}
info: [tigerconnect] Thread: pt:cffbe708 (prefix: HH, category: patient-specific)
{
    "timestamp": "2026-07-09T16:49:36.941Z"
}
info: [tigerconnect] Images: 1 fetched (29KB), 0 skipped, 598ms
{
    "timestamp": "2026-07-09T16:49:40.341Z"
}
info: [tigerconnect] Extracted 27 messages (27 with absolute date, 1 attachments)
{
    "timestamp": "2026-07-09T16:49:40.341Z"
}
info: [tigerconnect] Found 15 members
{
    "timestamp": "2026-07-09T16:49:46.141Z"
}
info: [tigerconnect] Checking HH HOSPITALIZATION/FALLS thread
{
    "timestamp": "2026-07-09T16:49:47.242Z"
}
info: [click] Search Icon
{
    "timestamp": "2026-07-09T16:49:53.740Z"
}
warn: [tigerconnect] Search panel did not open, trying evaluate click
{
    "timestamp": "2026-07-09T16:49:57.740Z"
}
warn: [retry] Open search attempt 1/3 failed: Search panel still not visible after evaluate click
{
    "timestamp": "2026-07-09T16:50:02.139Z"
}
info: [sessionPool] Closed cmqk0f98y0001thyexv93hbje:careficient (idle)
{
    "timestamp": "2026-07-09T16:50:07.043Z"
}
info: [click] Search Icon
{
    "timestamp": "2026-07-09T16:50:09.040Z"
}
info: [tigerconnect] Typed "HOSPITALIZATION" — results ready (has-results, 1999ms)
{
    "timestamp": "2026-07-09T16:50:25.640Z"
}
info: [tigerconnect] done (Puppeteer) — 2 findings
{
    "timestamp": "2026-07-09T16:50:26.941Z"
}
info: [audit] extract tigerconnect ok
{
    "action": "extract",
    "actor": "agent",
    "audit": "v1",
    "durationMs": 321168,
    "meta": {
        "failCount": 2,
        "transport": "puppeteer"
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "ok",
    "target": "puppeteer",
    "timestamp": "2026-07-09T16:50:26.942Z",
    "ts": "2026-07-09T16:50:26.942Z",
    "vendor": "tigerconnect"
}
warn: [launcher] Browser disconnected (process exit, crash, or remote WS drop)
{
    "timestamp": "2026-07-09T16:50:29.342Z"
}
info: [launcher] Browser closed
{
    "timestamp": "2026-07-09T16:50:31.441Z"
}
error: [Analyzer] Analysis failed: Expected ',' or ']' after array element in JSON at position 7323
{
    "timestamp": "2026-07-09T16:51:12.443Z"
}
info: [audit] llm.analyzer  error
{
    "action": "llm.analyzer",
    "actor": "agent",
    "audit": "v1",
    "durationMs": 41002,
    "meta": {
        "errorMessage": "Expected ',' or ']' after array element in JSON at position 7323"
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "error",
    "target": null,
    "timestamp": "2026-07-09T16:51:12.443Z",
    "ts": "2026-07-09T16:51:12.443Z",
    "vendor": null
}
info: [run] Results posted for P-1782913193302-2
{
    "timestamp": "2026-07-09T16:51:12.719Z"
}
info: [run] Extraction complete for P-1782913193302-2
{
    "timestamp": "2026-07-09T16:51:12.719Z"
}
info: [audit] run.end  partial
{
    "action": "run.end",
    "actor": "agent",
    "audit": "v1",
    "durationMs": null,
    "meta": {
        "hasAnalysis": false,
        "systemSummary": {
            "inovalon": "ok",
            "tigerconnect": "ok",
            "vonage": "error"
        }
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "partial",
    "target": null,
    "timestamp": "2026-07-09T16:51:12.719Z",
    "ts": "2026-07-09T16:51:12.719Z",
    "vendor": null
}
info: [server] Run complete for patient=P-1782913193302-2 in 474.6s
{
    "timestamp": "2026-07-09T16:51:12.720Z"
}
"""

# --- Run and print ---
clean_context = sanitize_log_for_llm(raw_aws_log_text)
print(clean_context)

POST /run 
{
    "patientId": "P-1782913193302-2",
    "systems": [
        "careficient"
    ],
    "timestamp": ""
}
Starting run for patient=P-1782913193302-2 (1/4 global, 1/2 org) 
[run] Starting Stagehand extraction for patient=P-1782913193302-2 (pt:a48892d3) systems=[careficient] 
[audit] run.start  ok 
{
    "action": "run.start",
    "actor": "agent",
    "audit": "v1",
    "durationMs": null,
    "meta": {
        "patientNamePresent": true,
        "systems": [
            "careficient"
        ]
    },
    "orgId": "cmqk0f98y0001thyexv93hbje",
    "patientId": "P-1782913193302-2",
    "result": "ok",
    "target": null,
    "timestamp": "",
    "ts": "",
    "vendor": null
}
[Stagehand] Initialized (LOCAL, model=bedrock/us.anthropic.claude-sonnet-4-6, act/extract retry-wrapped) 
[CF] In-memory extraction for pt:a48892d3 
[CF] Login attempt — username=present(17ch), password=present(14ch) 
[CF] Logged in (attempt 1/3) 
[CF] openPatient: deterministic search for pt:a48892d3 
[